In [1]:
import pandas as pd
import numpy as np
import json
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score, average_precision_score, confusion_matrix, f1_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
import preprocessor as p
from Rumors_Classifier.utils import parse_propagation_file
import emoji

In [2]:
def extract_features(text):
    # Use preprocessor to get structured entities
    parsed = p.parse(text)

    # 1. Basic Length Features
    char_len = len(text)  # Total characters
    word_len = len(text.split())  # Total words

    # 2. Extracted Entities
    hashtags = len(parsed.hashtags) if parsed.hashtags else 0
    mentions = len(parsed.mentions) if parsed.mentions else 0
    urls = len(parsed.urls) if parsed.urls else 0
    emojis = 0
    # Use the 'emoji' library for a more accurate emoji count
    for character in text:
        if emoji.is_emoji(character):
            emojis += 1

    # 3. Exclamation Count
    exclamations = text.count('!')

    return pd.Series([char_len, word_len, hashtags, mentions, urls, emojis, exclamations])

In [3]:
def prepare_data(tweets_path, replies_path, retweets_path):
    # Load
    tweets = pd.read_csv(tweets_path, sep='\t')
    tweets['tweetID'] = tweets['tweetID'].astype(str)

    # Text features
    p.set_options(p.OPT.URL, p.OPT.MENTION, p.OPT.HASHTAG, p.OPT.EMOJI)
    tweets[['char_len', 'word_len', 'num_hashtags', 'num_mentions', 'num_urls', 'num_emojis', 'num_exclamations']] = tweets['tweetText'].apply(extract_features)
    tweets['text_dup_count'] = tweets.groupby('tweetText')['tweetText'].transform('count')
    # Deduplicate
    tweets = tweets.drop_duplicates(subset=['tweetText'], keep='first')

    # Propagation
    reply_counts = parse_propagation_file(replies_path)
    retweet_counts = parse_propagation_file(retweets_path)
    tweets['num_replies'] = tweets['tweetID'].map(reply_counts).fillna(0).astype(int)
    tweets['num_retweets'] = tweets['tweetID'].map(retweet_counts).fillna(0).astype(int)

    # Average word length
    tweets['avg_word_length'] = tweets['char_len'] / tweets['word_len'].replace(0, np.nan)

    # Drop columns
    tweets = tweets.drop(['word_len'], axis=1, errors='ignore')

    return tweets

In [4]:
tweets_path ="../../ArCOV19-Rumors/tweet_verification/Tweets.txt"
replies_path='../../ArCOV19-Rumors/tweet_verification/propagation_networks/replies'
retweets_path='../../ArCOV19-Rumors/tweet_verification/propagation_networks/retweets'

tweets_df = prepare_data(tweets_path, replies_path, retweets_path)

In [6]:
from sklearn.model_selection import train_test_split

# Set random state for reproducibility
RANDOM_STATE = 42
TEST_SIZE = 0.2
VAL_SIZE = 0.2

# First split: separate test set
X_temp, X_test, y_temp, y_test = train_test_split(
    tweets_df, tweets_df['label'], test_size=TEST_SIZE, stratify=tweets_df['label'], random_state=RANDOM_STATE
)

# Then split the temporary set into train and validation (val = 0.25 of temp equals 20% of total)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=VAL_SIZE/(1-TEST_SIZE), stratify=y_temp, random_state=RANDOM_STATE
)

print(f"Train size: {len(X_train)}")
print(f"Validation size: {len(X_val)}")
print(f"Test size: {len(X_test)}")
print(f"Train label ratio: {y_train.mean():.3f}")
print(f"Val label ratio: {y_val.mean():.3f}")
print(f"Test label ratio: {y_test.mean():.3f}")

Train size: 2078
Validation size: 693
Test size: 693
Train label ratio: 0.507
Val label ratio: 0.506
Test label ratio: 0.506


In [7]:
features = ['tweetText', 'tweetID', 'char_len', 'avg_word_length', 'num_hashtags', 'num_mentions',
                     'num_urls', 'num_emojis', 'num_exclamations', 'num_replies', 'num_retweets']

handcrafted_cols = [
    'char_len', 'avg_word_length', 'num_hashtags', 'num_mentions',
    'num_urls', 'num_emojis', 'num_exclamations', 'num_replies', 'num_retweets'
]

# Define column transformer
preprocessor = ColumnTransformer([
    ('handcrafted', StandardScaler(), handcrafted_cols),
    ('tfidf', TfidfVectorizer(ngram_range=(1,2), max_features=5000, sublinear_tf=True), 'tweetText')
])

In [8]:
# scaling weights for imbalance
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_pos_weight = neg_count / pos_count if pos_count > 0 else 1.0
print(f"scale_pos_weight = {scale_pos_weight:.2f}")

xgb_model = XGBClassifier(
    objective='binary:logistic',
    scale_pos_weight=scale_pos_weight,
    n_estimators=2000,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    eval_metric='logloss',
    use_label_encoder=False,
    random_state=RANDOM_STATE,
    early_stopping_rounds=20,
)

X_train_transformed = preprocessor.fit_transform(X_train)
X_val_transformed = preprocessor.transform(X_val)

# Then train XGBoost with early stopping
eval_set = [(X_val_transformed, y_val)]
xgb_model.fit(
    X_train_transformed, y_train,
    eval_set=eval_set,
    verbose=True
)

# Now wrap the trained model and preprocessor together for saving
final_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', xgb_model)
])

scale_pos_weight = 0.97
[0]	validation_0-logloss:0.67980
[1]	validation_0-logloss:0.66786
[2]	validation_0-logloss:0.65600
[3]	validation_0-logloss:0.64437
[4]	validation_0-logloss:0.63532
[5]	validation_0-logloss:0.62592
[6]	validation_0-logloss:0.61517
[7]	validation_0-logloss:0.60759
[8]	validation_0-logloss:0.59867
[9]	validation_0-logloss:0.59058
[10]	validation_0-logloss:0.58339
[11]	validation_0-logloss:0.57734


C:\Users\LENOVO\Personal Projects\NLP\arabic_rumor_scanner\ARS_venv\Lib\site-packages\xgboost\callback.py:385: UserWarning: [15:25:43] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  self.starting_round = model.num_boosted_rounds()


[12]	validation_0-logloss:0.57110
[13]	validation_0-logloss:0.56568
[14]	validation_0-logloss:0.55882
[15]	validation_0-logloss:0.55311
[16]	validation_0-logloss:0.54797
[17]	validation_0-logloss:0.54323
[18]	validation_0-logloss:0.53819
[19]	validation_0-logloss:0.53268
[20]	validation_0-logloss:0.52813
[21]	validation_0-logloss:0.52438
[22]	validation_0-logloss:0.52083
[23]	validation_0-logloss:0.51656
[24]	validation_0-logloss:0.51308
[25]	validation_0-logloss:0.50964
[26]	validation_0-logloss:0.50530
[27]	validation_0-logloss:0.50171
[28]	validation_0-logloss:0.49885
[29]	validation_0-logloss:0.49607
[30]	validation_0-logloss:0.49325
[31]	validation_0-logloss:0.48971
[32]	validation_0-logloss:0.48669
[33]	validation_0-logloss:0.48368
[34]	validation_0-logloss:0.48153
[35]	validation_0-logloss:0.47780
[36]	validation_0-logloss:0.47601
[37]	validation_0-logloss:0.47233
[38]	validation_0-logloss:0.46962
[39]	validation_0-logloss:0.46629
[40]	validation_0-logloss:0.46322
[41]	validatio

In [9]:
X_test_transformed = preprocessor.transform(X_test)
y_pred_proba = xgb_model.predict_proba(X_test_transformed)[:, 1]
y_pred = (y_pred_proba >= 0.5).astype(int)


accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)
pr_auc = average_precision_score(y_test, y_pred_proba)
cm = confusion_matrix(y_test, y_pred)

print("\n=== Baseline XGBoost (Combined Features) ===")
print(f"Accuracy:  {accuracy:.4f}")
print(f"F1-macro:  {f1:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")
print(f"PR-AUC:    {pr_auc:.4f}")
print(f"Confusion Matrix:\n{cm}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['False', 'True']))

# Save metrics to JSON
metrics = {
    'accuracy': accuracy,
    'f1_macro': f1,
    'roc_auc': roc_auc,
    'pr_auc': pr_auc,
    'confusion_matrix': cm.tolist(),
    'train_size': len(X_train),
    'val_size': len(X_val),
    'test_size': len(X_test),
    'scale_pos_weight': scale_pos_weight,
    'early_stopping_rounds': xgb_model.best_iteration,
    'best_score': xgb_model.best_score
}

with open('logs/baseline_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("Metrics saved to logs/baseline_metrics.json")


=== Baseline XGBoost (Combined Features) ===
Accuracy:  0.8672
F1-macro:  0.8667
ROC-AUC:   0.9434
PR-AUC:    0.9500
Confusion Matrix:
[[302  40]
 [ 52 299]]

Classification Report:
              precision    recall  f1-score   support

       False       0.85      0.88      0.87       342
        True       0.88      0.85      0.87       351

    accuracy                           0.87       693
   macro avg       0.87      0.87      0.87       693
weighted avg       0.87      0.87      0.87       693

Metrics saved to logs/baseline_metrics.json


In [10]:
import joblib

# Save the full pipeline (including preprocessor and trained classifier)
joblib.dump(final_pipeline, 'models/xgb_combined_pipeline.pkl')
print("Model saved to models/xgb_combined_pipeline.pkl")

# Also save just the preprocessor separately (optional, for inference flexibility)
joblib.dump(preprocessor, 'models/preprocessor.pkl')

Model saved to models/xgb_combined_pipeline.pkl


['models/preprocessor.pkl']